In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sys
import os

project_root = os.path.dirname(os.path.abspath(''))
if project_root not in sys.path:
    sys.path.append(project_root)

from FEATURES.features import *
from FEATURES.featuresV2 import *
from NOTEBOOKS.calculateEVS import *
from MODELS.pipeline import *
from MODELS.teamInfo import teamStarPlayer, projectedStartingFive

### Load Model

In [9]:
model = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_MODEL.pkl')
features = joblib.load('../MODELS/SAVED_MODELS/feature_list.pkl')

### Load Player Data and Bookmaker Data

In [10]:
pd.set_option('display.max_columns', None)
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

s25= pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_25.csv').sort_values(by='GAME_DATE')
s26 = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_26.csv').sort_values(by='GAME_DATE')
df = pd.concat([s25, s26])

usData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_US_{today}.csv')
dfsData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_DFS_{today}.csv')

### Update projected starting lineups

In [11]:
from MODELS.scrapStarting import NBADailyLineups

scraper = NBADailyLineups("https://www.rotowire.com/basketball/nba-lineups.php")
scraper.getDict()  # Scrape the lineups
scraper.updateTeamInfo()  # Update teamInfo.py

Successfully updated /Users/alexg/Documents/Documents/Prize-Picks-Prop-Predictor/MODELS/teamInfo.py
Updated 12 teams with confirmed lineups


### Top EVs for single bets

In [12]:
singlePTSBookies = usData[(usData['CATEGORY'] == 'player_points') & (usData['ODDS'] <= 200) & (usData['ODDS'] >= -200)]

results = calculateSingleBets(df, singlePTSBookies, model, features, current_date, edge_threshold=0.20, stake=10, 
                     variance_inflation=1.1, distribution_type='t', stat_col='PTS', 
                     use_monte_carlo=True, n_simulations=10000, max_kelly=0.25)

singleBets = results
singleBets = singleBets[(singleBets['SIGMA FLAG'] == 'Med') | (singleBets['SIGMA FLAG'] == 'Low')].sort_values(by='EV%', ascending=False).reset_index(drop=True)
singleBets.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/singleBets.csv', index=False)
singleBets.head()

Processing single bets with single model...


,NAME,BOOKMAKER,CATEGORY,LINE,ODDS,SIDE,PREDICTION,RECOMMENDATION,OVER%,UNDER%,IMPLIED PROB,MODEL PROB,EDGE,EV%,KELLY_FRACTION,KELLY_DOLLARS,CONFIDENCE INTERVAL,INTERVAL WIDTH,SIGMA,SIGMA FLAG,EXPECTED ROI,SIMULATION_METHOD
0,Goga Bitadze,BetMGM,player_points,4.5,105,Over,11.21,1,0.911,0.089,0.488,0.911,0.423,8.67,0.826,2.62,"(0.3, 22.1)",21.73,5.54,Med,0.87,Monte Carlo
1,Giannis Antetokounmpo,Bovada,player_points,29.5,125,Under,26.48,1,0.193,0.807,0.444,0.807,0.363,8.16,0.653,3.12,"(18.4, 34.6)",16.17,4.13,Low,0.82,Monte Carlo
2,Giannis Antetokounmpo,Bovada,player_points,28.5,150,Under,26.48,1,0.275,0.725,0.400,0.725,0.325,8.13,0.542,3.75,"(18.4, 34.6)",16.17,4.13,Low,0.81,Monte Carlo
3,Giannis Antetokounmpo,Bovada,player_points,30.5,105,Under,26.48,1,0.133,0.867,0.488,0.867,0.379,7.78,0.741,2.62,"(18.4, 34.6)",16.17,4.13,Low,0.78,Monte Carlo
4,Giannis Antetokounmpo,Bovada,player_points,27.5,180,Under,26.48,1,0.377,0.623,0.357,0.623,0.266,7.45,0.414,4.50,"(18.4, 34.6)",16.17,4.13,Low,0.74,Monte Carlo


## Top EVs for 2 leg bets

### Underdog picks

In [13]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

results = calculate2LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=0.35, stake=100, 
                     variance_inflation=1.1, distribution_type='t',
                     use_monte_carlo=True, n_simulations=10000, max_kelly=0.25)

underdogPairs = results
underdogPairs = underdogPairs[
    underdogPairs[['sigma_flag1', 'sigma_flag2']].isin(['Med', 'Low']).all(axis=1)
].sort_values(by='ev_percent', ascending=False).reset_index(drop=True)
underdogPairs.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogPairs.csv', index=False)
underdogPairs.head()

Error getting prediction for Nique Clifford: single positional indexer is out-of-bounds
Error getting prediction for Nique Clifford: single positional indexer is out-of-bounds
Error getting prediction for Nique Clifford: single positional indexer is out-of-bounds
Error getting prediction for Nique Clifford: single positional indexer is out-of-bounds
Error getting prediction for Nique Clifford: single positional indexer is out-of-bounds
Error getting prediction for Nique Clifford: single positional indexer is out-of-bounds
Error getting prediction for Nique Clifford: single positional indexer is out-of-bounds
Error getting prediction for Nique Clifford: single positional indexer is out-of-bounds
Error getting prediction for Nique Clifford: single positional indexer is out-of-bounds
Error getting prediction for Nique Clifford: single positional indexer is out-of-bounds
Error getting prediction for Nique Clifford: single positional indexer is out-of-bounds
Error getting prediction for Niq

,player1,player2,line1,line2,pred1,pred2,model_side1,model_side2,prob1,prob2,prob_both,edge1,edge2,combined_edge,ev_percent,kelly_full,recommendation,confidence_interval1,confidence_interval2,interval_width1,interval_width2,sigma1,sigma2,sigma_flag1,sigma_flag2,simulation_method
0,Giannis Antetokounmpo,Stephen Curry,31.5,25.5,26.48,26.92,under,over,0.911,0.608,0.4988,0.333,0.03,0.182,0.5,0.248,0,"(18.4, 34.6)","(15.5, 38.4)",16.17,22.91,4.13,5.84,Low,Med,Monte Carlo


### Prizepicks picks

In [14]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

results = calculate2LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=0.35, stake=100, 
                     variance_inflation=1.1, distribution_type='t',
                     use_monte_carlo=True, n_simulations=10000, max_kelly=0.25)
pairsPrizepicks = results
pairsPrizepicks = pairsPrizepicks[
    pairsPrizepicks[['sigma_flag1', 'sigma_flag2']].isin(['Med', 'Low']).all(axis=1)
].sort_values(by='ev_percent', ascending=False).reset_index(drop=True)
pairsPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksPairs.csv', index=False)
pairsPrizepicks.head()

Error getting prediction for Nique Clifford: single positional indexer is out-of-bounds
Error getting prediction for Nique Clifford: single positional indexer is out-of-bounds
Error getting prediction for Nique Clifford: single positional indexer is out-of-bounds
Error getting prediction for Nique Clifford: single positional indexer is out-of-bounds
Error getting prediction for Nique Clifford: single positional indexer is out-of-bounds
Error getting prediction for Nique Clifford: single positional indexer is out-of-bounds
Error getting prediction for Nique Clifford: single positional indexer is out-of-bounds
Error getting prediction for Nique Clifford: single positional indexer is out-of-bounds
Error getting prediction for Nique Clifford: single positional indexer is out-of-bounds
Error getting prediction for Nique Clifford: single positional indexer is out-of-bounds
Error getting prediction for Nique Clifford: single positional indexer is out-of-bounds
Error getting prediction for Niq

,player1,player2,line1,line2,pred1,pred2,model_side1,model_side2,prob1,prob2,prob_both,edge1,edge2,combined_edge,ev_percent,kelly_full,recommendation,confidence_interval1,confidence_interval2,interval_width1,interval_width2,sigma1,sigma2,sigma_flag1,sigma_flag2,simulation_method
0,Giannis Antetokounmpo,Al Horford,31.5,6.5,26.48,9.23,under,over,0.911,0.719,0.5897,0.333,0.141,0.237,0.77,0.385,0,"(18.4, 34.6)","(0.0, 20.1)",16.17,20.11,4.13,5.55,Low,Med,Monte Carlo
1,Giannis Antetokounmpo,Stephen Curry,31.5,25.5,26.48,26.92,under,over,0.911,0.608,0.4988,0.333,0.030,0.182,0.50,0.248,0,"(18.4, 34.6)","(15.5, 38.4)",16.17,22.91,4.13,5.84,Low,Med,Monte Carlo


## 3 leg parlay

### Underdog picks

In [15]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

threeLeg = calculate3LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=0.10, stake=100, 
                     variance_inflation=1.1, distribution_type='t', 
                     use_monte_carlo=True, n_simulations=10000, max_kelly=0.25)

underdogTrios = threeLeg
underdogTrios = threeLeg[
    threeLeg[['sigma_flag1', 'sigma_flag2', 'sigma_flag3']].isin(['Med', 'Low']).all(axis=1)
].sort_values(by='ev_percent', ascending=False).reset_index(drop=True)
underdogTrios.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogTrios.csv', index=False)
underdogTrios.head()

Error getting prediction for Nique Clifford: single positional indexer is out-of-bounds
Error getting prediction for Nique Clifford: single positional indexer is out-of-bounds
Error getting prediction for Nique Clifford: single positional indexer is out-of-bounds
Error getting prediction for Nique Clifford: single positional indexer is out-of-bounds
Error getting prediction for Nique Clifford: single positional indexer is out-of-bounds
Error getting prediction for Nique Clifford: single positional indexer is out-of-bounds
Error getting prediction for Nique Clifford: single positional indexer is out-of-bounds
Error getting prediction for Nique Clifford: single positional indexer is out-of-bounds
Error getting prediction for Nique Clifford: single positional indexer is out-of-bounds
Error getting prediction for Nique Clifford: single positional indexer is out-of-bounds
Error getting prediction for Nique Clifford: single positional indexer is out-of-bounds
Error getting prediction for Niq

,player1,player2,player3,line1,line2,line3,pred1,pred2,pred3,model_side1,model_side2,model_side3,prob1,prob2,prob3,prob_all_three,edge1,edge2,edge3,combined_edge,ev_percent,kelly_full,recommendation,confidence_interval1,confidence_interval2,confidence_interval3,interval_width1,interval_width2,interval_width3,sigma1,sigma2,sigma3,sigma_flag1,sigma_flag2,sigma_flag3,simulation_method


### Prizepicks picks

In [16]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

threeLeg = calculate3LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=0.40, stake=100, 
                     variance_inflation=1.1, distribution_type='normal', 
                     use_monte_carlo=True, n_simulations=10000, max_kelly=0.25)

triosPrizepicks = threeLeg  
triosPrizepicks = threeLeg[
    threeLeg[['sigma_flag1', 'sigma_flag2', 'sigma_flag3']].isin(['Med', 'Low']).all(axis=1)
].sort_values(by='ev_percent', ascending=False).reset_index(drop=True)
triosPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksTrios.csv', index=False)
triosPrizepicks.head()

Error getting prediction for Nique Clifford: single positional indexer is out-of-bounds
Error getting prediction for Nique Clifford: single positional indexer is out-of-bounds
Error getting prediction for Nique Clifford: single positional indexer is out-of-bounds
Error getting prediction for Nique Clifford: single positional indexer is out-of-bounds
Error getting prediction for Nique Clifford: single positional indexer is out-of-bounds
Error getting prediction for Nique Clifford: single positional indexer is out-of-bounds
Error getting prediction for Nique Clifford: single positional indexer is out-of-bounds
Error getting prediction for Nique Clifford: single positional indexer is out-of-bounds
Error getting prediction for Nique Clifford: single positional indexer is out-of-bounds
Error getting prediction for Nique Clifford: single positional indexer is out-of-bounds
Error getting prediction for Nique Clifford: single positional indexer is out-of-bounds
Error getting prediction for Niq

,player1,player2,player3,line1,line2,line3,pred1,pred2,pred3,model_side1,model_side2,model_side3,prob1,prob2,prob3,prob_all_three,edge1,edge2,edge3,combined_edge,ev_percent,kelly_full,recommendation,confidence_interval1,confidence_interval2,confidence_interval3,interval_width1,interval_width2,interval_width3,sigma1,sigma2,sigma3,sigma_flag1,sigma_flag2,sigma_flag3,simulation_method
0,Giannis Antetokounmpo,Stephen Curry,Al Horford,31.5,25.5,6.5,26.48,26.92,9.23,under,over,over,0.888,0.6,0.684,0.2954,0.31,0.022,0.106,0.146,0.77,0.154,0,"(18.4, 34.6)","(15.5, 38.4)","(0.0, 20.1)",16.17,22.91,20.11,4.13,5.84,5.55,Low,Med,Med,Monte Carlo


In [17]:
def count_line_hits(player_df, line, category, game_windows=[5, 10, 15]):
    results = {}
    player_df_sorted = player_df.sort_values('GAME_DATE')
    total_games = len(player_df_sorted)

    for window in game_windows:
        # Handle players with fewer games
        if total_games < window:
            last_n_games = player_df_sorted
        else:
            last_n_games = player_df_sorted.tail(window)

        if category == 'player_points':
            hits = (last_n_games['PTS'] > line).sum()
        elif category == 'player_assists':
            hits = (last_n_games['AST'] > line).sum()
        elif category == 'player_rebounds':
            hits = (last_n_games['REB'] > line).sum()
        elif category == 'player_threes':
            hits = (last_n_games['FG3M'] > line).sum()
        elif category == 'player_blocks':
            hits = (last_n_games['BLK'] > line).sum()
        elif category == 'player_steals':
            hits = (last_n_games['STL'] > line).sum()
        elif category == 'player_field_goals':
            hits = (last_n_games['FGM'] > line).sum()
        elif category == 'player_frees_made':
            hits = (last_n_games['FTM'] > line).sum()
        elif category == 'player_points_rebounds_assists':
            hits = (last_n_games['PTS'] + last_n_games['REB'] + last_n_games['AST'] > line).sum()
        elif category == 'player_points_rebounds':
            hits = (last_n_games['PTS'] + last_n_games['REB'] > line).sum()
        elif category == 'player_points_assists':
            hits = (last_n_games['PTS'] + last_n_games['AST'] > line).sum()
        elif category == 'player_rebounds_assists':
            hits = (last_n_games['REB'] + last_n_games['AST'] > line).sum()
        elif category == 'player_turnovers':
            hits = (last_n_games['TOV'] > line).sum()
        else:
            hits = 0

        results['NAME'] = player_df_sorted['PLAYER_NAME'].iloc[0] if total_games > 0 else 'Unknown'
        results['CATEGORY'] = category
        results['LINE'] = line
        results[f'LAST {window}'] = hits
        results[f'HIT RATE % LAST {window}'] = round(hits / window, 2)


    return results

prizePicks = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks')]
prizePicks= prizePicks.drop_duplicates(subset=['CATEGORY', 'NAME', 'LINE'], keep='first')
prizePicks

,BOOKMAKER,CATEGORY,NAME,OVER/UNDER,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE
148,PrizePicks,player_points,Bobby Portis,Over,9.5,-137,2025-11-01,2025-11-01T18:19:33Z
152,PrizePicks,player_points,Kyle Kuzma,Over,9.0,-137,2025-11-01,2025-11-01T18:19:33Z
154,PrizePicks,player_points,Giannis Antetokounmpo,Over,31.5,-137,2025-11-01,2025-11-01T18:19:33Z
156,PrizePicks,player_points,Zach LaVine,Over,24.0,-137,2025-11-01,2025-11-01T18:19:33Z
158,PrizePicks,player_points,DeMar DeRozan,Over,20.5,-137,2025-11-01,2025-11-01T18:19:33Z
...,...,...,...,...,...,...,...,...
2244,PrizePicks,player_blocks_steals,Josh Minott,Over,1.5,-137,2025-11-02,2025-11-01T18:19:36Z
2246,PrizePicks,player_blocks_steals,Kevin Durant,Over,1.5,-137,2025-11-02,2025-11-01T18:19:36Z
2248,PrizePicks,player_blocks_steals,Tari Eason,Over,1.5,-137,2025-11-02,2025-11-01T18:19:36Z
2250,PrizePicks,player_blocks_steals,Josh Okogie,Over,1.5,-137,2025-11-02,2025-11-01T18:19:36Z


In [18]:
line_hit_data = []

for index, row in prizePicks.iterrows():
    name = row['NAME']
    line = row['LINE']
    category = row['CATEGORY']
    player_df = df[df['PLAYER_NAME'] == name]
    
    if len(player_df) > 0:
        hit_counts = count_line_hits(player_df, line, category)
        line_hit_data.append(hit_counts)

line_hit_df = pd.DataFrame(line_hit_data)



over_rates_dir = '../DATA/CSV_FILES/PROP_DATA/OVER_RATES_PRIZEPICKS'
os.makedirs(over_rates_dir, exist_ok=True)
today = datetime.today().strftime('%Y%m%d')

for category in line_hit_df['CATEGORY'].unique():
    # Filter data for this category
    category_data = line_hit_df[line_hit_df['CATEGORY'] == category]
    
    filename = f"{category}.csv"
    filepath = os.path.join(over_rates_dir, filename)
    category_data.to_csv(filepath, index=False)
    print(f"Saved {len(category_data)} records for {category} to {filename}")

print(f"\nAll category files saved to {over_rates_dir}")

        
        

Saved 79 records for player_points to player_points.csv
Saved 56 records for player_rebounds to player_rebounds.csv
Saved 33 records for player_assists to player_assists.csv
Saved 10 records for player_threes to player_threes.csv
Saved 10 records for player_blocks to player_blocks.csv
Saved 15 records for player_steals to player_steals.csv
Saved 17 records for player_field_goals to player_field_goals.csv
Saved 24 records for player_frees_made to player_frees_made.csv
Saved 81 records for player_points_rebounds_assists to player_points_rebounds_assists.csv
Saved 78 records for player_points_rebounds to player_points_rebounds.csv
Saved 73 records for player_points_assists to player_points_assists.csv
Saved 38 records for player_rebounds_assists to player_rebounds_assists.csv
Saved 17 records for player_turnovers to player_turnovers.csv
Saved 13 records for player_blocks_steals to player_blocks_steals.csv

All category files saved to ../DATA/CSV_FILES/PROP_DATA/OVER_RATES_PRIZEPICKS


In [19]:
underdog = dfsData[(dfsData['BOOKMAKER'] == 'Underdog')]
underdog= underdog.drop_duplicates(subset=['CATEGORY', 'NAME', 'LINE'], keep='first')
line_hit_data = []

for index, row in underdog.iterrows():
    name = row['NAME']
    line = row['LINE']
    category = row['CATEGORY']
    player_df = df[df['PLAYER_NAME'] == name]
    
    if len(player_df) > 0:
        hit_counts = count_line_hits(player_df, line, category)
        line_hit_data.append(hit_counts)

line_hit_df = pd.DataFrame(line_hit_data)



over_rates_dir = '../DATA/CSV_FILES/PROP_DATA/OVER_RATES_UNDERDOG'
os.makedirs(over_rates_dir, exist_ok=True)
today = datetime.today().strftime('%Y%m%d')

for category in line_hit_df['CATEGORY'].unique():
    # Filter data for this category
    category_data = line_hit_df[line_hit_df['CATEGORY'] == category]
    
    filename = f"{category}.csv"
    filepath = os.path.join(over_rates_dir, filename)
    category_data.to_csv(filepath, index=False)
    print(f"Saved {len(category_data)} records for {category} to {filename}")

print(f"\nAll category files saved to {over_rates_dir}")

Saved 66 records for player_points to player_points.csv
Saved 18 records for player_rebounds to player_rebounds.csv
Saved 12 records for player_assists to player_assists.csv
Saved 8 records for player_threes to player_threes.csv
Saved 1 records for player_steals to player_steals.csv
Saved 3 records for player_frees_made to player_frees_made.csv
Saved 79 records for player_points_rebounds_assists to player_points_rebounds_assists.csv
Saved 31 records for player_points_rebounds to player_points_rebounds.csv
Saved 25 records for player_points_assists to player_points_assists.csv
Saved 11 records for player_rebounds_assists to player_rebounds_assists.csv
Saved 4 records for player_turnovers to player_turnovers.csv
Saved 1 records for player_blocks_steals to player_blocks_steals.csv

All category files saved to ../DATA/CSV_FILES/PROP_DATA/OVER_RATES_UNDERDOG
